In [1]:
# Import required system
import sys
sys.path.append('/Users/hoto7260/LIET/Jacob_LIET2/LIET/liet')
sys.path.append('/Users/hoto7260/LIET/Hope_Graphing_LIET/functions')

# general
import matplotlib.pyplot as plt
import numpy as np
try:
    np.distutils.__config__.blas_opt_info = np.distutils.__config__.blas_ilp64_opt_info
except Exception:
    pass
import pandas as pd
from scipy.stats import pearsonr
import plotly.express as px

# my modules
from liet_res_class import FitParse
import rnap_lib_data_proc  as dp
import plotting_funcs as pf
import analysis_funcs as af

# setting plotting paramaters
from pylab import rcParams
rcParams['figure.figsize'] = 20, 6
rcParams['font.size'] = 15
#plt.rcParams.update({'font.size': 22})

/tmp/ipykernel_357497/1732951080.py:13: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [21]:
# for each gene, get the proper locations
def get_LIET_end_results(LIET_df, endtype = "0.9_perc"):
    """
    Parameters
    ----------
    LIET_df: the LIET df output by FitParse in plotting_functions.py
    end_type: str
        - either EMG, Full, or another string (string must be a column in the dataframe)
        IN ALL CASES assumes the positive strand is the sense strand
    This function adds additional BID specific related elements to the dataframe: calculates the end positions for both strands.
    in: pandas Dataframe from the FitParse dataframe class with the sL, tI, and start information
    out: pandas DataFrame with the following ADDITIONAL columns (one row per bidirectional):
        * rel_pos_end = sL_mean + tI_mean
        * rel_pos_end_interval = (sL_mean - sL_stdev)+(tI_mean - tI_stdev),(sL_mean + sL_stdev)+(tI_mean + tI_stdev)
        * abs_pos_end = start + sL_mean + tI_mean
        * abs_pos_end_interval = start+[(sL_mean - sL_stdev)+(tI_mean - tI_stdev)],start+[(sL_mean + sL_stdev)+(tI_mean + tI_stdev)]
        * rel_neg_end = sL_a_mean + tI_a_mean
        * rel_neg_end_interval = (sL_a_mean - sL_a_stdev)+(tI_a_mean - tI_a_stdev),(sL_a_mean + sL_a_stdev)+(tI_a_mean + tI_a_stdev)
        * abs_neg_end = mu - sL_a_mean - tI_a_mean
        * abs_neg_end_interval = start+[(sL_a_mean - sL_a_stdev)+(tI_a_mean - tI_a_stdev)],start+[(sL_a_mean + sL_a_stdev)+(tI_a_mean + tI_a_stdev)]
    """
    # if mT then use that
    if endtype == "mT":
        # Positive
        LIET_df = LIET_df.assign(rel_pos_end = LIET_df["mT_mean"].astype(int)) 
        LIET_df = LIET_df.assign(abs_pos_end = LIET_df["start"]+LIET_df["mT_mean"].astype(int))
        LIET_df = LIET_df.assign(rel_neg_end = LIET_df["mT_a_mean"].astype(int)) 
        LIET_df = LIET_df.assign(abs_neg_end = LIET_df["start"]-LIET_df["mT_a_mean"].astype(int))
    else:
        # SENSE STRAND
        pos_end = "_".join([endtype, "pos"])
        LIET_df = LIET_df.assign(rel_pos_end = LIET_df[pos_end]) 
        LIET_df = LIET_df.assign(abs_pos_end = LIET_df["start"]+LIET_df[pos_end])
        # negative strand in my case
        neg_end = "_".join([endtype, "neg"])
        LIET_df = LIET_df.assign(rel_neg_end = LIET_df[neg_end]) 
        LIET_df = LIET_df.assign(abs_neg_end = LIET_df["start"]-LIET_df[neg_end])
    return LIET_df


def get_LIET_end_bed(LIET_df, end_type, antisense=True):
    """
    This function transforms the FitParse dataframe of bidirectionals into a bed file where the start position
    refers to the 3' most end of the - transcript and end to the 3' most end of the + transcript 
    (midpoint is NOT mu). Mu is saved as a separate column. The df also contains additional
    information that might be helpful for downstream analysis or evaluating the trustworthiness of the results.
    in: pandas Dataframe from the FitParse dataframe class that has undergone the get_LIET_end_results function.
    out: pandas DataFrame with the following columns (one row per bidirectional):
        * chr
        * start = most upstream (3' end of - strand)
        * end = most downstream (3' end of + strand)
        * name = name with |- or |+ according to strand
        * wLI_mean = w_LI_mean,w_aLI_mean
        * coverage = pos_cov,neg_cov
        * LI_coverage = coverage attributable to NOT background
        * rel_end_stdev = rel_pos_end_stdev,rel_neg_end_stdev
        * mu
        * elbo_lrange
    """
    if end_type == "EMG":
        LI_pos_cov = LIET_df["pos_cov"]*LIET_df["w_LI_mean"]
        rel_pos_end_stdev = LIET_df["sL_stdev"]+LIET_df["tI_stdev"]
        if antisense:
            LI_neg_cov = LIET_df["neg_cov"]*LIET_df["w_aLI_mean"]
            rel_neg_end_stdev = LIET_df["sL_a_stdev"]+LIET_df["tI_a_stdev"]
        else:
            LI_neg_cov = ["NA"]*len(LIET_df.chrom)
            rel_neg_end_stdev = ["NA"]*len(LIET_df.chrom)
        bed = pd.DataFrame({"chr": LIET_df.chrom, 
                               "start": LIET_df.start+LIET_df.mL_mean.astype(int), "end": LIET_df.abs_pos_end.astype(int), 
                              "name": LIET_df["gene"] , 
                              "wLI_mean": LIET_df["w_LI_mean"].astype(str)+","+LIET_df["w_aLI_mean"].astype(str),
                            "coverage": LIET_df["pos_cov"].astype(str)+","+LIET_df["neg_cov"].astype(str), 
                            "LI_coverage": LI_pos_cov.astype(int).astype(str)+","+LI_neg_cov.astype(int).astype(str),
                               "rel_end_stdev": rel_pos_end_stdev.round(1).astype(str)+","+rel_neg_end_stdev.round(1).astype(str), 
                           "mu":LIET_df.start, "elbow_lrange": LIET_df.elbo_lrange})
    else:
        if antisense:
            print("Not available rn")
        else:
            if end_type == "Full":
                rel_end_stdev=LIET_df["mT_stdev"].round(1).astype(str)
            else:
                rel_end_stdev= ["NA"]*len(LIET_df.chrom)
            non_wB_mean = 1 - LIET_df["w_B_mean"]
            LIE_pos_cov = LIET_df["pos_cov"]*non_wB_mean
            bed = pd.DataFrame({"chr": LIET_df.chrom, 
                                   "start": LIET_df.start+LIET_df.mL_mean.astype(int), "end": LIET_df.abs_pos_end.astype(int), 
                                  "name":  LIET_df["gene"]+"|+" , 
                                "LIET_coverage": LIE_pos_cov.astype(int),
                                "strand": ["+"]*LIET_df.shape[0],
                                "coverage": LIET_df["pos_cov"].astype(str), 
                                  "wLIET_mean": non_wB_mean.round(3),
                                "filler": ["."]*LIET_df.shape[0],
                                   "rel_end_stdev": rel_end_stdev, 
                               "mu":LIET_df.start, "elbow_lrange": LIET_df.elbo_lrange})
        
    return bed

def get_LIET_end_sep_bed(LIET_df, ET_sense=True, antisense=True, ET_antisense=False):
    """
    This function transforms the FitParse dataframe of bidirectionals into a bed file where the start position
    refers to the 3' most end of the - transcript and end to the 3' most end of the + transcript 
    (midpoint is NOT mu). Mu is saved as a separate column. The df also contains additional
    information that might be helpful for downstream analysis or evaluating the trustworthiness of the results.
    in: pandas Dataframe from the FitParse dataframe class that has undergone the get_LIET_end_results function.
    out: pandas DataFrame with the following columns (two rows per bidirectional):
        * chr
        * start = most upstream (3' end of - strand & 5' end of + strand)
        * end = most downstream (3' end of + strand & 5' end of - strand)
        * name = name with |- or |+ according to strand
        * wLI_mean = w_LI_mean or w_aLI_mean
        * coverage = pos_cov or neg_cov
        * LI_coverage = coverage attributable to NOT background
        * rel_end_stdev = rel_pos_end_stdev or rel_neg_end_stdev
        * mu
        * elbo_lrange (same for both strands)
    """
    # get the postive strand bed
    if ET_sense:
        w_LIET = 1-LIET_df["w_B_mean"]
        LIET_pos_cov = LIET_df["pos_cov"]*w_LIET
        pos_bed = pd.DataFrame({"chr": LIET_df.chrom, 
                               "start": LIET_df.start+LIET_df.mL_mean.astype(int), "end": LIET_df.abs_pos_end.astype(int), 
                              "name": LIET_df["gene"]+"|+" , 
                                "LIET_coverage": LIET_pos_cov.astype(int),
                                "strand": ["+"]*LIET_df.shape[0],
                            "coverage": LIET_df["pos_cov"], 
                                "wLIET_mean": w_LIET.round(3),
                                "filler": ["."]*LIET_df.shape[0],
                           "mu":LIET_df.start, "elbow_lrange": LIET_df.elbo_lrange})
    else:
        LI_pos_cov = LIET_df["pos_cov"]*LIET_df["w_LI_mean"]
        pos_bed = pd.DataFrame({"chr": LIET_df.chrom, 
                               "start": LIET_df.start+LIET_df.mL_mean.astype(int), "end": LIET_df.abs_pos_end.astype(int), 
                              "name": LIET_df["gene"]+"|+" , 
                                "LI_coverage": LI_pos_cov.astype(int),
                                "strand": ["+"]*LIET_df.shape[0],
                            "coverage": LIET_df["pos_cov"], 
                                "wLI_mean": LIET_df["w_LI_mean"].round(3),
                                "filler": ["."]*LIET_df.shape[0],
                           "mu":LIET_df.start, "elbow_lrange": LIET_df.elbo_lrange})
    # get the negative strand bed
    if ET_antisense:
        w_a_LIET = 1-LIET_df["w_aB_mean"]
        LIET_neg_cov = LIET_df["neg_cov"]*w_a_LIET
        neg_bed = pd.DataFrame({"chr": LIET_df.chrom, 
                               "start": LIET_df.abs_neg_end.astype(int), "end": LIET_df.start - LIET_df.mL_a_mean.astype(int), 
                              "name": LIET_df["gene"]+"|-" , 
                                "LIET_coverage": LIET_neg_cov.astype(int),
                                "strand": ["-"]*LIET_df.shape[0],
                            "coverage": LIET_df["neg_cov"], 
                                "wLIET_mean": w_a_LIET.round(3),
                                "filler": ["."]*LIET_df.shape[0],
                           "mu":LIET_df.start, "elbow_lrange": LIET_df.elbo_lrange})
    else:
        
        LI_neg_cov = LIET_df["neg_cov"]*LIET_df["w_aLI_mean"]
        neg_bed = pd.DataFrame({"chr": LIET_df.chrom, 
                               "start": LIET_df.abs_neg_end.astype(int), "end": LIET_df.start - LIET_df.mL_a_mean.astype(int), 
                              "name": LIET_df["gene"]+"|-" , 
                              "LI_coverage": LI_neg_cov.astype(int),
                                "strand": ["-"]*LIET_df.shape[0],
                            "coverage": LIET_df["neg_cov"], 
                                "wLI_mean": LIET_df["w_aLI_mean"].round(3),
                                "filler": ["."]*LIET_df.shape[0],
                           "mu":LIET_df.start, "elbow_lrange": LIET_df.elbo_lrange})
    # combine the two
    return pd.concat([pos_bed, neg_bed])
    


## Save the stranded bed files 
* To use other percentiles, change "0.95_perc" to "0.85_perc", etc.
* To use mT values, change "0.95_perc" to "mT". **Using mT requires that you ran the full model**

**Colon_Format Option**
* People often name enhancers according to chromomsome and positions separated by a colon. Many parameters are also separated by a colon in the results file. To address this, if you named your regions with a colon included, have colon_format=True otherwise you will get an error.

In [50]:
prefixes = ["sm36-ALI-D21_120UPM-1_EMG", "sm36-ALI-D21_120UPM-2_EMG", 
           "sm36-ALI-D21_30UPM-1_EMG", "sm36-ALI-D21_30UPM-2_EMG", 
           "sm36-ALI-D21_veh-1_EMG", "sm36-ALI-D21_veh-2_EMG"]

prefixes = ["sm36-ALI-D21_30UPM-2_EMG", "SRR13772353_EMG"]
bed_dir="/scratch/Users/hoto7260/Resp_Env/Comb_UPM_WSP_ADP/LIET/LIET/LIET_results/beds"
from pathlib import Path


In [51]:
# save dataframe results in a dictionary
df_dict = dict()
for prefix in prefixes:
    print(prefix)
    ## Original files
    log_file = out_dir+prefix+"/"+prefix+".liet.log"
    res_file = out_dir+prefix+"/"+prefix+".liet"
    results = FitParse(res_file=res_file, log_file=log_file, antisense=True, 
                                     ET_sense=False, ET_antisense=False, colon_format=True)
    ends = get_LIET_end_results(results.df, endtype = "0.95_perc")
    bed = get_LIET_end_sep_bed(ends, ET_sense=False, antisense=True, ET_antisense=False)

    ## Now error files (only if they exist)
    log_file = out_dir+prefix+"_err/"+prefix+".liet.log"
    res_file = out_dir+prefix+"_err/"+prefix+".liet"
    if Path(res_file).is_file():
        err_results = FitParse(res_file=res_file, log_file=log_file, antisense=True, 
                                         ET_sense=False, ET_antisense=False, colon_format=True)
        ends = get_LIET_end_results(err_results.df, endtype = "0.95_perc")
        err_bed = get_LIET_end_sep_bed(ends, ET_sense=False, antisense=True, ET_antisense=False)
        # combine
        full_bed = pd.concat([bed, err_bed])
        full_results_df = pd.concat([results.df, err_results.df])
    else:
        full_bed = bed
        full_results_df = results.df
    df_dict[prefix] = full_results_df
    full_bed.to_csv("".join([prefix, "_95p.liet.bed"]), 
                sep="\t", header=False, index=False)
        

sm36-ALI-D21_30UPM-2_EMG


CHECK LINE: ['chr13:105981024: fitting error']
CHECK LINE: ['chr13:109213060: model error']
CHECK LINE: ['chr13:109647311: model error']
CHECK LINE: ['chr13:110012222: model error']
CHECK LINE: ['chr13:110715565: model error']
CHECK LINE: ['chr14:22614289: fitting error']
CHECK LINE: ['chr14:24274334: model error']
CHECK LINE: ['chr14:32078045: model error']
CHECK LINE: ['chr14:35365599: fitting error']
CHECK LINE: ['chr14:36856477: model error']
CHECK LINE: ['chr14:37571271: model error']
CHECK LINE: ['chr14:38808927: model error']
CHECK LINE: ['chr14:51479947: model error']
CHECK LINE: ['chr14:55339862: model error']
CHECK LINE: ['chr14:68527525: model error']
CHECK LINE: ['chr14:69658124: fitting error']
CHECK LINE: ['chr14:70707402: model error']
CHECK LINE: ['chr14:75868193: model error']
CHECK LINE: ['chr14:77981073: model error']
CHECK LINE: ['chr14:81401428: fitting error']
CHECK LINE: ['chr14:90513810: fitting error']
CHECK LINE: ['chr14:94396118: fitting error']
CHECK LINE: [

Number of features considered: 3083
SRR13772353_EMG


CHECK LINE: ['chr1:149847302: model error']
CHECK LINE: ['chr1:151680892: model error']
CHECK LINE: ['chr12:45233041: fitting error']
CHECK LINE: ['chr14:53947013: fitting error']
CHECK LINE: ['chr17:3049015: model error']
CHECK LINE: ['chr18:3246234: fitting error']
CHECK LINE: ['chr19:3369654: model error']
CHECK LINE: ['chr2:45650050: model error']
CHECK LINE: ['chr2:121075328: model error']
CHECK LINE: ['chr2:218733759: fitting error']
CHECK LINE: ['chr2:237673700: fitting error']
CHECK LINE: ['chr3:12188166: model error']
CHECK LINE: ['chr3:138633880: model error']
CHECK LINE: ['chr6:108592766: model error']
CHECK LINE: ['chr7:6635481: fitting error']
CHECK LINE: ['chr7:12210619: model error']
CHECK LINE: ['chr7:16754428: model error']
CHECK LINE: ['chr7:91266394: model error']
CHECK LINE: ['chr7:106660063: model error']
CHECK LINE: ['chr8:98940128: model error']
CHECK LINE: ['chr9:79572578: model error']
CHECK LINE: ['chr9:111693805: model error']
CHECK LINE: ['chr9:113341856: mo

Number of features considered: 4476
Number of features considered: 13


CHECK LINE: ['chr1:149847302: fitting error']
CHECK LINE: ['chr19:3369654: fitting error']
CHECK LINE: ['chr7:12210619: fitting error']
CHECK LINE: ['chr9:115591791: fitting error']
CHECK LINE: ['chr17:3049015: fitting error']
CHECK LINE: ['chr2:45650050: fitting error']
CHECK LINE: ['chr2:237673700: fitting error']
CHECK LINE: ['chr6:108592766: fitting error']
CHECK LINE: ['chr7:16754428: fitting error']
CHECK LINE: ['chr8:98940128: fitting error']
CHECK LINE: ['chr9:128120516: fitting error']
CHECK LINE: ['chr12:45233041: fitting error']
CHECK LINE: ['chr18:3246234: fitting error']


## Getting conensus calls from multiple samples
* Use a weighted average where the call is weighted by the amount of NON-background coverage

In [54]:
print(df_dict[prefix].columns)

Index(['chrom', 'start', 'stop', 'strand', 'gene', 'mL_mean', 'mL_stdev',
       'sL_mean', 'sL_stdev', 'tI_mean', 'tI_stdev', 'w_LI_mean', 'w_LI_stdev',
       'w_B_mean', 'w_B_stdev', 'mL_a_mean', 'mL_a_stdev', 'sL_a_mean',
       'sL_a_stdev', 'tI_a_mean', 'tI_a_stdev', 'w_aLI_mean', 'w_aLI_stdev',
       'w_aB_mean', 'w_aB_stdev', '0.75_perc_pos', '0.75_perc_neg',
       '0.8_perc_pos', '0.8_perc_neg', '0.85_perc_pos', '0.85_perc_neg',
       '0.9_perc_pos', '0.9_perc_neg', '0.95_perc_pos', '0.95_perc_neg',
       'pos_cov', 'neg_cov', 'elbo_lrange', 'elbo_urange', 'fit_time_min'],
      dtype='object')


In [81]:
# Combine all dataframes in the dictionary
combined = pd.concat(
    [df.assign(source=key) for key, df in df_dict.items()],
    ignore_index=True
)
combined["pos_cov_nob"] = (1 - combined["w_B_mean"])* combined['pos_cov']
combined["neg_cov_nob"] = abs((1 - combined["w_aB_mean"])* combined['neg_cov'])

def compute_weighted(group):
    """
    Get the weighted average of the end. Here the end is based on the 95 percentile.
    The coverage (not including background) is used to weight the calls where the calls with higher coverage are weighted higher.
    """
    total_pos_cov = group['pos_cov_nob'].sum()
    total_neg_cov = group['neg_cov_nob'].sum()
    
    # Weighted averages (protect against division by zero)
    weighted_pos = (group['0.95_perc_pos'] * group['pos_cov_nob']).sum() / total_pos_cov if total_pos_cov > 0 else float('nan')
    weighted_neg = (group['0.95_perc_neg'] * group['neg_cov_nob']).sum() / total_neg_cov if total_neg_cov > 0 else float('nan')
    
    # Number of unique dataframes that had this gene
    num_dfs = group['source'].nunique()
    
    # Average coverage
    avg_pos_cov = group['pos_cov_nob'].mean()
    avg_neg_cov = group['neg_cov_nob'].mean()

    return pd.Series({
        'weighted_0.95_perc_pos': weighted_pos,
        'weighted_0.95_perc_neg': weighted_neg,
        'num_dataframes': num_dfs,
        'avg_pos_cov_nob': avg_pos_cov,
        'avg_neg_cov_nob': avg_neg_cov
    })

consensus = combined.groupby('gene').apply(compute_weighted, include_groups=False).reset_index()
consensus.iloc[0:3,]

,gene,weighted_0.95_perc_pos,weighted_0.95_perc_neg,num_dataframes,avg_pos_cov_nob,avg_neg_cov_nob
0,chr10:100311438,683.000000,205.000000,1.0,23.40,13.02
1,chr10:100346521,1385.302398,875.057179,2.0,626.26,510.68
2,chr10:100348799,NaN,1414.929936,2.0,0.00,50.24


## I want to look at details of my calls
The FitParse class produces a dataframe that stores all of the relevant values: 
1. Original annotation file inputs: 'chrom', 'start', 'stop', 'strand', 'gene',
2. Final mean and standard deviations of the posterior position/sigma values (in bp and if position relative to start:
    *  Sense (Positive if ENhancer): 'mL_mean', 'mL_stdev', 'sL_mean', 'sL_stdev', 'tI_mean', 'tI_stdev'
    *  Antisense (Negative if Enhancer): 'mL_a_mean', 'mL_a_stdev', 'sL_a_mean', 'sL_a_stdev', 'tI_a_mean', 'tI_a_stdev'
3. Final mean standard deviations of the weights of each part of the model (LI and B for ET=False and LI, E, T, and B for ET=True)
    * Sense (Positive if enhancer): 'w_LI_mean', 'w_LI_stdev', 'w_B_mean', 'w_B_stdev'
    * Antisense (Negative if enhancer): 'w_aLI_mean', 'w_aLI_stdev', 'w_aB_mean', 'w_aB_stdev'
4. Percentile Positions (in bp relative to start, if _neg then to left of start)
    * _pos = sense/positive strand, _neg = antisense/negative strand
    * '0.75_perc_pos', '0.75_perc_neg','0.8_perc_pos', '0.8_perc_neg', '0.85_perc_pos', '0.85_perc_neg',
       '0.9_perc_pos', '0.9_perc_neg', '0.95_perc_pos', '0.95_perc_neg'
5. Coverage: Number of reads in the full padded region on positive and negative strand
    * 'pos_cov', 'neg_cov',
6. Elbow Loss Range (Lower and Upper)
    * 'elbo_lrange', 'elbo_urange', 
8. Time used to fit the model
    * 'fit_time_min'

In [85]:
print(df_dict[prefix].columns)

Index(['chrom', 'start', 'stop', 'strand', 'gene', 'mL_mean', 'mL_stdev',
       'sL_mean', 'sL_stdev', 'tI_mean', 'tI_stdev', 'w_LI_mean', 'w_LI_stdev',
       'w_B_mean', 'w_B_stdev', 'mL_a_mean', 'mL_a_stdev', 'sL_a_mean',
       'sL_a_stdev', 'tI_a_mean', 'tI_a_stdev', 'w_aLI_mean', 'w_aLI_stdev',
       'w_aB_mean', 'w_aB_stdev', '0.75_perc_pos', '0.75_perc_neg',
       '0.8_perc_pos', '0.8_perc_neg', '0.85_perc_pos', '0.85_perc_neg',
       '0.9_perc_pos', '0.9_perc_neg', '0.95_perc_pos', '0.95_perc_neg',
       'pos_cov', 'neg_cov', 'elbo_lrange', 'elbo_urange', 'fit_time_min',
       'pos_cov_nob', 'neg_cov_nob'],
      dtype='object')


In [86]:
df_dict[prefix].iloc[0:3,]

,chrom,start,stop,strand,gene,mL_mean,mL_stdev,sL_mean,sL_stdev,tI_mean,...,0.9_perc_neg,0.95_perc_pos,0.95_perc_neg,pos_cov,neg_cov,elbo_lrange,elbo_urange,fit_time_min,pos_cov_nob,neg_cov_nob
chr10:100346521,chr10,100346521,100346721,1,chr10:100346521,32.74,18.74,404.15,19.19,114.39,...,749,841,940,280,-848,8434.010605,10146.755816,4.17,274.4,780.16
chr10:100348799,chr10,100348799,100348999,1,chr10:100348799,3.89,70.24,68.29,74.04,50.74,...,173,200,212,2048,-36,17746.375715,23787.155931,8.03,0.0,1.08
chr10:100997734,chr10,100997734,100997934,1,chr10:100997734,32.46,28.45,280.73,26.35,75.47,...,608,588,778,1750,-2241,34046.224232,44009.411948,13.68,140.0,112.05
